# 19.12 — Membership Inference

Membership inference asks a simple privacy question: given a trained model, an example `(x, y)`, and the model's score on that example, can an attacker tell whether that example was in the training set? In this lesson we build the whole audit from scratch in NumPy: a toy classifier, the train/test loss gap that creates leakage, a loss-threshold attack, shadow models that choose the threshold without touching the victim's data, and a likelihood-ratio attack (LiRA) that compares how surprising a score is under member versus non-member behavior.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build membership inference one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is made visible so the attack is an auditable calculation rather than a black box. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, random simulation, and linear algebra for the toy privacy audit.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for data, model fitting, and attack splits.

### 1. A toy classifier whose training examples get lower loss

Membership inference starts with a model quantity that behaves differently on training points than on fresh points. We will use per-example logistic loss, because it is small when the model assigns high probability to the true label and large when the model is uncertain or wrong. A model that fits its training set too tightly often gives members unusually low losses — an overfitting fingerprint.

In [ ]:
def sigmoid_w(z_w):
    return 1 / (1 + np.exp(-np.clip(z_w, -30, 30)))

def make_blobs_w(n_w, seed_w, noise_w=0.9):
    rng_w = np.random.default_rng(seed_w)
    y_w = rng_w.integers(0, 2, size=n_w)
    centers_w = np.where(y_w[:, None] == 1, np.array([1.0, 1.0]), np.array([-1.0, -1.0]))
    X_w = centers_w + noise_w * rng_w.normal(size=(n_w, 2))
    return X_w, y_w.astype(float)

X_train_w, y_train_w = make_blobs_w(80, 1)
X_test_w, y_test_w = make_blobs_w(400, 2)
print("train shape:", X_train_w.shape, "test shape:", X_test_w.shape)
print("train class rate:", round(float(y_train_w.mean()), 3))

▶ What you'll see: a small training set and a larger fresh test set drawn from the same two-class toy distribution.

In [ ]:
def fit_logreg_w(X_w, y_w, steps_w=2500, lr_w=0.25, lam_w=0.001):
    X1_w = np.c_[np.ones(len(X_w)), X_w]
    theta_w = np.zeros(X1_w.shape[1])
    for _ in range(steps_w):
        p_w = sigmoid_w(X1_w @ theta_w)
        grad_w = X1_w.T @ (p_w - y_w) / len(y_w) + lam_w * np.r_[0, theta_w[1:]]
        theta_w -= lr_w * grad_w
    return theta_w

def predict_proba_w(X_w, theta_w):
    return sigmoid_w(np.c_[np.ones(len(X_w)), X_w] @ theta_w)

theta_w = fit_logreg_w(X_train_w, y_train_w)
train_p_w = predict_proba_w(X_train_w, theta_w)
test_p_w = predict_proba_w(X_test_w, theta_w)
print("theta:", np.round(theta_w, 3))
print("train accuracy:", round(float(np.mean((train_p_w >= 0.5) == y_train_w)), 3))
print("test accuracy:", round(float(np.mean((test_p_w >= 0.5) == y_test_w)), 3))

▶ What you'll see: a useful classifier whose train and test accuracies are similar but not identical.

In [ ]:
def log_loss_w(p_w, y_w):
    eps_w = 1e-12
    p_w = np.clip(p_w, eps_w, 1 - eps_w)
    return -(y_w * np.log(p_w) + (1 - y_w) * np.log(1 - p_w))

train_loss_w = log_loss_w(train_p_w, y_train_w)
test_loss_w = log_loss_w(test_p_w, y_test_w)
print("mean train loss:", round(float(train_loss_w.mean()), 3))
print("mean test loss:", round(float(test_loss_w.mean()), 3))
assert train_loss_w.mean() < test_loss_w.mean()

▶ What you'll see: the training loss is lower than the test loss, which is exactly the separation a membership attacker tries to exploit.

In [ ]:
plt.figure(figsize=(5, 3))
plt.hist(train_loss_w, bins=22, alpha=0.65, label="members: train loss", color="seagreen")
plt.hist(test_loss_w, bins=22, alpha=0.55, label="non-members: test loss", color="crimson")
plt.xlabel("per-example logistic loss")
plt.ylabel("count")
plt.title("1: member vs non-member loss histograms")
plt.legend()
plt.show()

▶ What you'll see: the member histogram leans left. The overlap is why the attack is imperfect; the left shift is why the attack is possible.

*Why it's done this way:* Logistic loss is a calibrated way to inspect confidence on the true label: for a correct label with probability `p`, the loss is `-log(p)`, so moving from `p=0.9` to `p=0.99` visibly lowers loss. Training minimizes average loss on members, so overfitting naturally creates a loss gap even when raw accuracy looks fine. The attacker does not need to know the true training algorithm's internals; it only needs a score whose distribution differs for members and non-members.

### 2. The loss-threshold attack and attack advantage

The simplest attack is the rule from the source lesson: $$\hat m=\mathbf{1}[s(x,y)\ge \tau]$$. If we let the score be negative loss, `s = -loss`, then high score means "the model is unusually confident on this labeled example." Equivalently, we can threshold loss directly and call low-loss points members.

In [ ]:
member_scores_w = -train_loss_w
nonmember_scores_w = -test_loss_w
all_scores_w = np.r_[member_scores_w, nonmember_scores_w]
all_m_w = np.r_[np.ones_like(member_scores_w), np.zeros_like(nonmember_scores_w)]
print("member score mean:", round(float(member_scores_w.mean()), 3))
print("non-member score mean:", round(float(nonmember_scores_w.mean()), 3))

▶ What you'll see: members have larger scores because their losses are smaller.

In [ ]:
thresholds_w = np.linspace(all_scores_w.min(), all_scores_w.max(), 200)
best_adv_w, best_tau_w, best_acc_w = -1, None, None
for tau_w in thresholds_w:
    pred_m_w = (all_scores_w >= tau_w).astype(float)
    tpr_w = np.mean(pred_m_w[all_m_w == 1] == 1)
    fpr_w = np.mean(pred_m_w[all_m_w == 0] == 1)
    adv_w = tpr_w - fpr_w
    acc_w = np.mean(pred_m_w == all_m_w)
    if adv_w > best_adv_w:
        best_adv_w, best_tau_w, best_acc_w = adv_w, tau_w, acc_w
print("best score threshold tau:", round(float(best_tau_w), 3))
print("attack accuracy:", round(float(best_acc_w), 3))
print("attack advantage TPR-FPR:", round(float(best_adv_w), 3))
assert best_adv_w > 0.10

▶ What you'll see: the threshold attack beats random guessing by a measurable advantage.

In [ ]:
tpr_curve_w, fpr_curve_w = [], []
for tau_w in thresholds_w:
    pred_m_w = (all_scores_w >= tau_w).astype(float)
    tpr_curve_w.append(np.mean(pred_m_w[all_m_w == 1] == 1))
    fpr_curve_w.append(np.mean(pred_m_w[all_m_w == 0] == 1))
order_w = np.argsort(fpr_curve_w)
auc_w = float(np.trapz(np.array(tpr_curve_w)[order_w], np.array(fpr_curve_w)[order_w]))
print("ROC AUC:", round(auc_w, 3))
assert auc_w > 0.53
plt.figure(figsize=(4.5, 3.5))
plt.plot(np.array(fpr_curve_w)[order_w], np.array(tpr_curve_w)[order_w], color="navy", label=f"AUC={auc_w:.3f}")
plt.plot([0, 1], [0, 1], "--", color="gray", label="random")
plt.xlabel("false positive rate")
plt.ylabel("true positive rate")
plt.title("2: ROC for the loss-threshold attack")
plt.legend()
plt.show()

▶ What you'll see: the ROC curve sits above the diagonal. Look for how far it bows upward; that distance is privacy leakage.

*Why it's done this way:* Raw attack accuracy can be misleading when membership labels are imbalanced, so `TPR - FPR` measures the attack's advantage over a false-alarm baseline. The threshold is chosen on attack calibration data, not on the victim point being judged, because otherwise the attacker would overfit the audit itself. ROC repeats the same threshold rule over all possible `τ` values, showing the full privacy tradeoff instead of one cherry-picked operating point.

### 3. Shadow models estimate the threshold without seeing victim membership

A realistic attacker may not know which examples were in the victim model's training set. Shadow models solve that calibration problem: train many lookalike models on data from the same population, record each shadow model's losses on its own training data (known members) and held-out data (known non-members), and choose an attack threshold from those labeled shadow losses.

In [ ]:
def shadow_losses_w(seed_w):
    Xs_w, ys_w = make_blobs_w(80, seed_w)
    Xh_w, yh_w = make_blobs_w(200, seed_w + 1000)
    th_w = fit_logreg_w(Xs_w, ys_w, steps_w=1400, lr_w=0.25, lam_w=0.001)
    in_loss_w = log_loss_w(predict_proba_w(Xs_w, th_w), ys_w)
    out_loss_w = log_loss_w(predict_proba_w(Xh_w, th_w), yh_w)
    return in_loss_w, out_loss_w

shadow_in_w, shadow_out_w = [], []
for seed_w in range(10, 18):
    a_w, b_w = shadow_losses_w(seed_w)
    shadow_in_w.append(a_w)
    shadow_out_w.append(b_w)
shadow_in_w = np.concatenate(shadow_in_w)
shadow_out_w = np.concatenate(shadow_out_w)
print("shadow member losses:", shadow_in_w.shape[0])
print("shadow non-member losses:", shadow_out_w.shape[0])

▶ What you'll see: the attacker has many labeled member/non-member loss samples from its own shadow experiments.

In [ ]:
shadow_scores_w = np.r_[-shadow_in_w, -shadow_out_w]
shadow_m_w = np.r_[np.ones_like(shadow_in_w), np.zeros_like(shadow_out_w)]
shadow_taus_w = np.linspace(shadow_scores_w.min(), shadow_scores_w.max(), 200)
shadow_adv_w = []
for tau_w in shadow_taus_w:
    pred_w = (shadow_scores_w >= tau_w).astype(float)
    shadow_adv_w.append(np.mean(pred_w[shadow_m_w == 1] == 1) - np.mean(pred_w[shadow_m_w == 0] == 1))
shadow_tau_w = float(shadow_taus_w[int(np.argmax(shadow_adv_w))])
print("shadow-chosen tau:", round(shadow_tau_w, 3))

▶ What you'll see: a threshold chosen without using the victim model's true membership labels.

In [ ]:
victim_pred_w = (all_scores_w >= shadow_tau_w).astype(float)
victim_tpr_w = np.mean(victim_pred_w[all_m_w == 1] == 1)
victim_fpr_w = np.mean(victim_pred_w[all_m_w == 0] == 1)
print("victim TPR:", round(float(victim_tpr_w), 3))
print("victim FPR:", round(float(victim_fpr_w), 3))
print("victim advantage:", round(float(victim_tpr_w - victim_fpr_w), 3))
assert victim_tpr_w - victim_fpr_w > 0.05
plt.figure(figsize=(5, 3))
plt.hist(shadow_in_w, bins=25, alpha=0.65, label="shadow members", color="seagreen")
plt.hist(shadow_out_w, bins=25, alpha=0.5, label="shadow non-members", color="crimson")
plt.axvline(-shadow_tau_w, color="black", linestyle="--", label="chosen loss cutoff")
plt.xlabel("loss")
plt.title("3: shadow losses calibrate the attack")
plt.legend()
plt.show()

▶ What you'll see: the vertical cutoff is learned from shadows, then transferred to the victim. It should land where member losses are relatively dense and non-member losses thin out.

*Why it's done this way:* Shadow models convert a privacy question with unknown labels into a supervised calibration problem. The attacker cannot train on the victim's membership labels, but it can imitate the data distribution and training recipe to estimate what member and non-member scores tend to look like. This matters because choosing `τ` on the victim audit set would inflate attack performance, exactly like tuning a model on its test set.

### 4. LiRA: a likelihood-ratio view of membership evidence

LiRA (Likelihood Ratio Attack) goes beyond one global threshold. Instead of asking only whether loss is below a cutoff, it asks: "is this score more likely under the member-score distribution or under the non-member-score distribution?" In the simplest Gaussian version, we estimate two normal distributions for the score `s=-loss` and compute a log likelihood ratio.

In [ ]:
def normal_logpdf_w(x_w, mu_w, sd_w):
    sd_w = max(float(sd_w), 1e-6)
    return -0.5 * np.log(2 * np.pi * sd_w ** 2) - 0.5 * ((x_w - mu_w) / sd_w) ** 2

mu_in_w, raw_sd_in_w = float(shadow_scores_w[shadow_m_w == 1].mean()), float(shadow_scores_w[shadow_m_w == 1].std())
mu_out_w, raw_sd_out_w = float(shadow_scores_w[shadow_m_w == 0].mean()), float(shadow_scores_w[shadow_m_w == 0].std())
sd_pool_w = float(np.sqrt((raw_sd_in_w ** 2 + raw_sd_out_w ** 2) / 2))
sd_in_w, sd_out_w = sd_pool_w, sd_pool_w
print("member score Gaussian: mean", round(mu_in_w, 3), "sd", round(sd_in_w, 3))
print("non-member score Gaussian: mean", round(mu_out_w, 3), "sd", round(sd_out_w, 3))
assert mu_in_w > mu_out_w

▶ What you'll see: the member-score distribution is shifted upward because members tend to have lower loss.

In [ ]:
llr_w = normal_logpdf_w(all_scores_w, mu_in_w, sd_in_w) - normal_logpdf_w(all_scores_w, mu_out_w, sd_out_w)
print("mean member LLR:", round(float(llr_w[all_m_w == 1].mean()), 3))
print("mean non-member LLR:", round(float(llr_w[all_m_w == 0].mean()), 3))
assert llr_w[all_m_w == 1].mean() > llr_w[all_m_w == 0].mean()

▶ What you'll see: members have larger log likelihood ratios, meaning their scores look more member-like under the shadow distributions.

In [ ]:
llr_taus_w = np.linspace(llr_w.min(), llr_w.max(), 200)
llr_tpr_w, llr_fpr_w = [], []
for tau_w in llr_taus_w:
    pred_w = (llr_w >= tau_w).astype(float)
    llr_tpr_w.append(np.mean(pred_w[all_m_w == 1] == 1))
    llr_fpr_w.append(np.mean(pred_w[all_m_w == 0] == 1))
llr_order_w = np.argsort(llr_fpr_w)
llr_auc_w = float(np.trapz(np.array(llr_tpr_w)[llr_order_w], np.array(llr_fpr_w)[llr_order_w]))
print("LiRA-style AUC:", round(llr_auc_w, 3))
assert llr_auc_w > 0.53
plt.figure(figsize=(4.5, 3.5))
plt.plot(np.array(llr_fpr_w)[llr_order_w], np.array(llr_tpr_w)[llr_order_w], color="purple", label=f"LiRA AUC={llr_auc_w:.3f}")
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("false positive rate")
plt.ylabel("true positive rate")
plt.title("4: ROC using likelihood-ratio scores")
plt.legend()
plt.show()

▶ What you'll see: a ROC curve from likelihood ratios. Look for whether it improves or matches the simple loss-threshold curve.

*Why it's done this way:* A likelihood ratio is the Bayesian evidence update in log form: positive values mean the observed score is more probable if the point was a member than if it was not. LiRA is powerful because it can use distribution shape and, in stronger versions, per-example or per-class calibration; a single threshold assumes all examples share the same score distribution. Here we keep the Gaussian version small so the math is visible: log density under "in" minus log density under "out."

### 5. Overfitting is the leak amplifier, and regularization is a privacy knob

The same attack can be rerun after changing the model's capacity or regularization. If a model memorizes more, member losses move farther left relative to non-member losses, and membership inference gets easier. If regularization closes the train-test loss gap, the attack surface shrinks.

In [ ]:
lams_w = np.array([0.0, 0.001, 0.02, 0.2])
gaps_w, aucs_w = [], []
for lam_w in lams_w:
    th_w = fit_logreg_w(X_train_w, y_train_w, steps_w=2200, lr_w=0.25, lam_w=float(lam_w))
    tr_loss_w = log_loss_w(predict_proba_w(X_train_w, th_w), y_train_w)
    te_loss_w = log_loss_w(predict_proba_w(X_test_w, th_w), y_test_w)
    scores_w = np.r_[-tr_loss_w, -te_loss_w]
    labels_w = np.r_[np.ones_like(tr_loss_w), np.zeros_like(te_loss_w)]
    taus_w = np.linspace(scores_w.min(), scores_w.max(), 150)
    tpr_w, fpr_w = [], []
    for tau_w in taus_w:
        pred_w = (scores_w >= tau_w).astype(float)
        tpr_w.append(np.mean(pred_w[labels_w == 1] == 1))
        fpr_w.append(np.mean(pred_w[labels_w == 0] == 1))
    ord_w = np.argsort(fpr_w)
    gaps_w.append(float(te_loss_w.mean() - tr_loss_w.mean()))
    aucs_w.append(float(np.trapz(np.array(tpr_w)[ord_w], np.array(fpr_w)[ord_w])))
print("loss gaps:", np.round(gaps_w, 3))
print("attack AUCs:", np.round(aucs_w, 3))
assert max(aucs_w) > 0.53

▶ What you'll see: different regularization strengths produce different loss gaps and different attack AUCs.

In [ ]:
fig, ax1 = plt.subplots(figsize=(5, 3))
ax1.plot(lams_w, gaps_w, marker="o", color="crimson", label="test loss - train loss")
ax1.set_xscale("symlog", linthresh=0.001)
ax1.set_xlabel("regularization λ")
ax1.set_ylabel("loss gap", color="crimson")
ax2 = ax1.twinx()
ax2.plot(lams_w, aucs_w, marker="s", color="navy", label="attack AUC")
ax2.set_ylabel("attack AUC", color="navy")
plt.title("5: overfitting gap and attack strength")
plt.show()

▶ What you'll see: attack strength tracks the separation between member and non-member losses, though tiny samples make the curve imperfect.

*Why it's done this way:* Membership inference is not a separate magic failure; it is often the privacy face of generalization failure. Empirical risk minimization optimizes members directly, so the more the model's behavior differs between optimized points and fresh points, the more evidence an attacker gets. Regularization, early stopping, data augmentation, and differential privacy all matter because they reduce how uniquely a single row can shape the released model.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, simulation, and small models built from scratch.
import matplotlib.pyplot as plt # load Matplotlib for loss histograms, ROC curves, and audit plots.
np.random.seed(0) # make the notebook's stochastic examples reproducible.

## 🟢 Basics (warm-up)

### Basic 1 — Turn model confidence into loss

**Goal.** Convert probabilities on true labels into per-example logistic losses, because membership attacks often start from unusually low training loss. We build it in 2 steps.

In [ ]:
p_b1 = np.array([0.95, 0.80, 0.55, 0.20]) # predicted probability assigned to label 1 for four examples.
y_b1 = np.array([1, 1, 0, 0]) # true labels for the same examples.
print("probabilities:", p_b1) # inspect the model scores before turning them into losses.
print("labels:", y_b1) # inspect the targets used to choose the correct probability.

In [ ]:
true_prob_b1 = np.where(y_b1 == 1, p_b1, 1 - p_b1) # select p for positive labels and 1-p for negative labels.
loss_b1 = -np.log(np.clip(true_prob_b1, 1e-12, 1.0)) # compute logistic loss on the true label.
print("true-label probabilities:", np.round(true_prob_b1, 3)) # inspect confidence on the correct class.
print("losses:", np.round(loss_b1, 3)) # inspect the attack score source.
assert round(float(loss_b1[0]), 3) == 0.051 # verify -log(0.95).
plt.figure(figsize=(4, 3)) # create a compact loss chart.
plt.bar(range(len(loss_b1)), loss_b1, color="teal") # show one loss per example.
plt.title("Basic 1: confidence becomes loss") # title the plot.
plt.xlabel("example") # label examples.
plt.ylabel("logistic loss") # label the loss scale.
plt.show() # display the chart.

▶ What you'll see: confident correct examples have tiny loss, while uncertain or wrong-looking examples have larger loss.

👀 Takeaway: a membership signal can come from the model being too confident on the true label.

### Basic 2 — Make a member/non-member score table

**Goal.** Build a labeled audit table with known membership labels, because attack evaluation needs both model scores and ground-truth membership. We build it in 2 steps.

In [ ]:
member_loss_b2 = np.array([0.05, 0.12, 0.18, 0.30, 0.42]) # toy losses on known training examples.
nonmember_loss_b2 = np.array([0.20, 0.35, 0.55, 0.80, 1.10]) # toy losses on held-out examples.
score_b2 = -np.r_[member_loss_b2, nonmember_loss_b2] # use negative loss so higher means more member-like.
m_b2 = np.r_[np.ones(len(member_loss_b2)), np.zeros(len(nonmember_loss_b2))] # 1=member, 0=non-member.
print("scores:", np.round(score_b2, 3)) # inspect the attack scores.
print("membership labels:", m_b2.astype(int)) # inspect the known labels.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact histogram.
plt.hist(-score_b2[m_b2 == 1], bins=5, alpha=0.65, label="members", color="seagreen") # show member losses.
plt.hist(-score_b2[m_b2 == 0], bins=5, alpha=0.55, label="non-members", color="crimson") # show non-member losses.
plt.title("Basic 2: labeled audit losses") # title the histogram.
plt.xlabel("loss") # label the loss axis.
plt.legend() # show group labels.
plt.show() # display the histogram.

▶ What you'll see: member losses are generally smaller, but the two groups overlap.

👀 Takeaway: membership inference is a binary classification problem over model-derived scores.

### Basic 3 — Apply one threshold rule

**Goal.** Implement `m_hat = 1[s >= τ]`, because the threshold attack is the simplest membership classifier. We build it in 2 steps.

In [ ]:
score_b3 = np.array([-0.05, -0.12, -0.18, -0.30, -0.42, -0.20, -0.35, -0.55, -0.80, -1.10]) # negative-loss scores.
m_b3 = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0]) # known membership labels for evaluation.
tau_b3 = -0.25 # classify examples with loss <= 0.25 as members.
pred_b3 = (score_b3 >= tau_b3).astype(int) # apply the membership rule.
print("predicted members:", pred_b3) # inspect the attack decisions.
print("true members:", m_b3) # inspect the ground truth.

In [ ]:
acc_b3 = np.mean(pred_b3 == m_b3) # compute raw attack accuracy.
tpr_b3 = np.mean(pred_b3[m_b3 == 1] == 1) # compute member detection rate.
fpr_b3 = np.mean(pred_b3[m_b3 == 0] == 1) # compute false alarms on non-members.
print("accuracy:", round(float(acc_b3), 3), "TPR:", round(float(tpr_b3), 3), "FPR:", round(float(fpr_b3), 3)) # inspect metrics.
assert round(float(acc_b3), 3) == 0.7 # verify this threshold's accuracy.
plt.figure(figsize=(4, 3)) # create a threshold plot.
plt.scatter(score_b3, m_b3, c=m_b3, cmap="coolwarm", s=70) # plot true membership by score.
plt.axvline(tau_b3, color="black", linestyle="--", label="τ") # show the threshold.
plt.title("Basic 3: threshold on score") # title the plot.
plt.xlabel("score = -loss") # label the score axis.
plt.ylabel("true membership") # label membership.
plt.legend() # show threshold label.
plt.show() # display the plot.

▶ What you'll see: examples to the right of the threshold are called members.

👀 Takeaway: the core attack is just a score, a threshold, and a binary decision.

### Basic 4 — Compute attack advantage

**Goal.** Measure `TPR - FPR`, because advantage reports how much better the attack is than false alarms. We build it in 2 steps.

In [ ]:
pred_b4 = np.array([1, 1, 1, 0, 0, 1, 0, 0, 0, 0]) # attack decisions from a toy threshold.
m_b4 = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0]) # true membership labels.
tp_b4 = np.sum((pred_b4 == 1) & (m_b4 == 1)) # count true positives.
fp_b4 = np.sum((pred_b4 == 1) & (m_b4 == 0)) # count false positives.
print("TP:", tp_b4, "FP:", fp_b4) # inspect numerator counts.

In [ ]:
tpr_b4 = tp_b4 / np.sum(m_b4 == 1) # member recall.
fpr_b4 = fp_b4 / np.sum(m_b4 == 0) # non-member false alarm rate.
adv_b4 = tpr_b4 - fpr_b4 # membership advantage.
print("TPR:", round(float(tpr_b4), 3), "FPR:", round(float(fpr_b4), 3), "advantage:", round(float(adv_b4), 3)) # inspect the privacy metric.
assert round(float(adv_b4), 3) == 0.4 # verify 0.6 - 0.2.
plt.figure(figsize=(4, 3)) # create a metric comparison chart.
plt.bar(["TPR", "FPR", "adv"], [tpr_b4, fpr_b4, adv_b4], color=["seagreen", "crimson", "navy"]) # show the pieces.
plt.title("Basic 4: attack advantage") # title the chart.
plt.ylim(0, 1) # keep the scale interpretable.
plt.show() # display the bars.

▶ What you'll see: advantage subtracts false alarms from member hits.

👀 Takeaway: privacy audits prefer advantage over accuracy when membership labels may be imbalanced.

### Basic 5 — Sweep thresholds and draw an ROC curve

**Goal.** Evaluate every threshold, because one cutoff hides the full privacy tradeoff between true positives and false positives. We build it in 3 steps.

In [ ]:
score_b5 = np.array([-0.05, -0.12, -0.18, -0.30, -0.42, -0.20, -0.35, -0.55, -0.80, -1.10]) # negative-loss scores.
m_b5 = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0]) # ground-truth membership labels.
taus_b5 = np.linspace(score_b5.min(), score_b5.max(), 50) # candidate thresholds.
print("threshold count:", len(taus_b5)) # inspect the sweep size.

In [ ]:
tpr_b5, fpr_b5 = [], [] # prepare ROC arrays.
for tau_b5 in taus_b5: # evaluate one threshold at a time.
    pred_b5 = (score_b5 >= tau_b5).astype(int) # call high-score examples members.
    tpr_b5.append(np.mean(pred_b5[m_b5 == 1] == 1)) # member hit rate.
    fpr_b5.append(np.mean(pred_b5[m_b5 == 0] == 1)) # non-member false alarm rate.
order_b5 = np.argsort(fpr_b5) # sort points from low to high FPR for AUC.
auc_b5 = float(np.trapz(np.array(tpr_b5)[order_b5], np.array(fpr_b5)[order_b5])) # approximate ROC area.
print("AUC:", round(auc_b5, 3)) # inspect threshold-free attack strength.
assert auc_b5 > 0.7 # verify clear leakage in the toy scores.

In [ ]:
plt.figure(figsize=(4, 3)) # create an ROC figure.
plt.plot(np.array(fpr_b5)[order_b5], np.array(tpr_b5)[order_b5], marker="o", color="purple") # draw ROC curve.
plt.plot([0, 1], [0, 1], "--", color="gray") # random-guess baseline.
plt.title("Basic 5: ROC sweep") # title the plot.
plt.xlabel("false positive rate") # label x-axis.
plt.ylabel("true positive rate") # label y-axis.
plt.show() # display the ROC curve.

▶ What you'll see: the curve rises above the diagonal when scores separate members from non-members.

👀 Takeaway: ROC summarizes all possible threshold attacks on the same score.

### Basic 6 — See why overfitting creates separation

**Goal.** Compare train and test losses directly, because the train-test gap is the attacker's raw material. We build it in 2 steps.

In [ ]:
train_loss_b6 = np.array([0.04, 0.08, 0.12, 0.20, 0.35, 0.50]) # losses on optimized training examples.
test_loss_b6 = np.array([0.15, 0.28, 0.40, 0.62, 0.90, 1.20]) # losses on fresh examples.
gap_b6 = float(test_loss_b6.mean() - train_loss_b6.mean()) # compute the generalization gap in loss.
print("train mean:", round(float(train_loss_b6.mean()), 3), "test mean:", round(float(test_loss_b6.mean()), 3)) # inspect means.
print("loss gap:", round(gap_b6, 3)) # inspect separation.

In [ ]:
plt.figure(figsize=(4, 3)) # create side-by-side bars.
plt.boxplot([train_loss_b6, test_loss_b6], labels=["train", "test"]) # summarize distributions.
plt.title("Basic 6: train-test loss gap") # title the plot.
plt.ylabel("loss") # label the loss scale.
plt.show() # display boxplots.
assert gap_b6 > 0.3 # verify a noticeable gap.

▶ What you'll see: training losses sit lower than test losses, giving the attacker a statistical cue.

👀 Takeaway: membership inference often measures overfitting from the privacy side.

### Basic 7 — Calibrate a threshold on shadow data

**Goal.** Choose `τ` from separate shadow losses, because the attack threshold should not be tuned on victim membership labels. We build it in 2 steps.

In [ ]:
shadow_in_b7 = np.array([0.05, 0.10, 0.18, 0.22, 0.31, 0.40]) # known member losses from shadow models.
shadow_out_b7 = np.array([0.16, 0.25, 0.38, 0.55, 0.72, 0.95]) # known non-member losses from shadows.
shadow_score_b7 = -np.r_[shadow_in_b7, shadow_out_b7] # convert to high-is-member scores.
shadow_m_b7 = np.r_[np.ones(len(shadow_in_b7)), np.zeros(len(shadow_out_b7))] # labels for calibration only.
print("shadow score range:", round(float(shadow_score_b7.min()), 3), "to", round(float(shadow_score_b7.max()), 3)) # inspect range.

In [ ]:
taus_b7 = np.linspace(shadow_score_b7.min(), shadow_score_b7.max(), 60) # candidate thresholds.
advs_b7 = [] # store shadow advantages.
for tau_b7 in taus_b7: # sweep thresholds.
    pred_b7 = (shadow_score_b7 >= tau_b7).astype(int) # classify with current threshold.
    advs_b7.append(np.mean(pred_b7[shadow_m_b7 == 1] == 1) - np.mean(pred_b7[shadow_m_b7 == 0] == 1)) # compute advantage.
best_tau_b7 = float(taus_b7[int(np.argmax(advs_b7))]) # select threshold by shadow advantage.
print("shadow-selected tau:", round(best_tau_b7, 3)) # inspect chosen threshold.
plt.figure(figsize=(4, 3)) # create calibration curve.
plt.plot(taus_b7, advs_b7, color="navy") # show advantage by threshold.
plt.axvline(best_tau_b7, color="red", linestyle="--") # mark selected threshold.
plt.title("Basic 7: shadow threshold calibration") # title the plot.
plt.xlabel("τ") # label threshold axis.
plt.ylabel("shadow advantage") # label metric.
plt.show() # display curve.

▶ What you'll see: one shadow threshold maximizes member/non-member separation on calibration data.

👀 Takeaway: shadow data lets the attacker tune the decision rule without peeking at victim labels.

### Basic 8 — Compute one likelihood ratio

**Goal.** Compare how likely one score is under member and non-member distributions, because LiRA is a likelihood-ratio attack. We build it in 2 steps.

In [ ]:
score_b8 = -0.12 # one candidate example's score, where larger means more member-like.
mu_in_b8, sd_in_b8 = -0.18, 0.10 # simple Gaussian model for member scores.
mu_out_b8, sd_out_b8 = -0.45, 0.20 # simple Gaussian model for non-member scores.
print("score:", score_b8) # inspect the candidate score.

In [ ]:
logpdf_in_b8 = -0.5 * np.log(2 * np.pi * sd_in_b8 ** 2) - 0.5 * ((score_b8 - mu_in_b8) / sd_in_b8) ** 2 # member log density.
logpdf_out_b8 = -0.5 * np.log(2 * np.pi * sd_out_b8 ** 2) - 0.5 * ((score_b8 - mu_out_b8) / sd_out_b8) ** 2 # non-member log density.
llr_b8 = logpdf_in_b8 - logpdf_out_b8 # log likelihood ratio.
print("log p(score|member):", round(float(logpdf_in_b8), 3)) # inspect member evidence.
print("log p(score|non-member):", round(float(logpdf_out_b8), 3)) # inspect non-member evidence.
print("LLR:", round(float(llr_b8), 3)) # inspect final evidence score.
assert llr_b8 > 0 # verify this score is more member-like.

▶ What you'll see: the score is more likely under the member Gaussian, so the LLR is positive.

👀 Takeaway: LiRA replaces a raw score with evidence: member likelihood divided by non-member likelihood.

### Basic 9 — Measure leakage under class imbalance

**Goal.** Compare accuracy and advantage when there are many non-members, because raw accuracy can hide a weak or trivial attack. We build it in 2 steps.

In [ ]:
m_b9 = np.r_[np.ones(10), np.zeros(90)] # imbalanced audit set with many more non-members.
pred_all_out_b9 = np.zeros_like(m_b9) # trivial attack that calls everyone non-member.
acc_all_out_b9 = np.mean(pred_all_out_b9 == m_b9) # raw accuracy of the trivial rule.
tpr_all_out_b9 = np.mean(pred_all_out_b9[m_b9 == 1] == 1) # member detection rate.
fpr_all_out_b9 = np.mean(pred_all_out_b9[m_b9 == 0] == 1) # false alarm rate.
print("trivial accuracy:", round(float(acc_all_out_b9), 3)) # inspect misleading accuracy.
print("trivial advantage:", round(float(tpr_all_out_b9 - fpr_all_out_b9), 3)) # inspect real attack value.

In [ ]:
pred_some_b9 = m_b9.copy() # create a stronger toy attack for comparison.
pred_some_b9[20:40] = 1 # add false positives among non-members.
acc_some_b9 = np.mean(pred_some_b9 == m_b9) # compute raw accuracy.
adv_some_b9 = np.mean(pred_some_b9[m_b9 == 1] == 1) - np.mean(pred_some_b9[m_b9 == 0] == 1) # compute advantage.
print("stronger accuracy:", round(float(acc_some_b9), 3), "stronger advantage:", round(float(adv_some_b9), 3)) # compare metrics.
plt.figure(figsize=(4, 3)) # create a metric chart.
plt.bar(["trivial acc", "trivial adv", "attack acc", "attack adv"], [acc_all_out_b9, 0, acc_some_b9, adv_some_b9], color=["gray", "gray", "teal", "teal"]) # show mismatch.
plt.xticks(rotation=20) # rotate labels for readability.
plt.title("Basic 9: imbalance changes metric meaning") # title the plot.
plt.show() # display chart.

▶ What you'll see: the trivial all-non-member rule has high accuracy but zero advantage.

👀 Takeaway: privacy leakage is about distinguishing members, not exploiting label imbalance.

### Basic 10 — Inspect a loss cutoff on real toy data

**Goal.** Generate a tiny train/test split and plot a cutoff, because visual inspection catches whether an attack is driven by real separation or noise. We build it in 3 steps.

In [ ]:
rng_b10 = np.random.default_rng(10) # local randomness for reproducibility.
member_loss_b10 = rng_b10.gamma(shape=1.5, scale=0.18, size=40) # synthetic low member losses.
nonmember_loss_b10 = rng_b10.gamma(shape=1.8, scale=0.28, size=120) # synthetic broader non-member losses.
cut_b10 = 0.28 # call examples with loss at or below this cutoff members.
print("member mean loss:", round(float(member_loss_b10.mean()), 3)) # inspect member losses.
print("non-member mean loss:", round(float(nonmember_loss_b10.mean()), 3)) # inspect non-member losses.

In [ ]:
pred_member_b10 = member_loss_b10 <= cut_b10 # member decisions for true members.
pred_nonmember_b10 = nonmember_loss_b10 <= cut_b10 # member decisions for true non-members.
tpr_b10 = float(pred_member_b10.mean()) # true positive rate.
fpr_b10 = float(pred_nonmember_b10.mean()) # false positive rate.
print("TPR:", round(tpr_b10, 3), "FPR:", round(fpr_b10, 3), "advantage:", round(tpr_b10 - fpr_b10, 3)) # inspect cutoff behavior.
assert tpr_b10 > fpr_b10 # verify the cutoff contains a membership signal.

In [ ]:
plt.figure(figsize=(5, 3)) # create loss histogram.
plt.hist(member_loss_b10, bins=18, alpha=0.65, label="members", color="seagreen") # plot member losses.
plt.hist(nonmember_loss_b10, bins=18, alpha=0.5, label="non-members", color="crimson") # plot non-member losses.
plt.axvline(cut_b10, color="black", linestyle="--", label="loss cutoff") # show threshold.
plt.title("Basic 10: inspect the cutoff") # title plot.
plt.xlabel("loss") # label x-axis.
plt.legend() # show labels.
plt.show() # display histogram.

▶ What you'll see: the cutoff slices through overlapping histograms; good audits report both hits and false alarms.

👀 Takeaway: always inspect member and non-member score distributions before trusting one privacy number.

## 🟡 Easy

### Easy 1 — Train a victim logistic model from scratch

**Goal.** Fit a NumPy-only classifier and measure the train-test loss gap, because membership inference needs a victim model with observable scores. We build it in 4 steps.

In [ ]:
rng_e1 = np.random.default_rng(21) # local generator for reproducible data.
y_train_e1 = rng_e1.integers(0, 2, size=70).astype(float) # training labels.
y_test_e1 = rng_e1.integers(0, 2, size=300).astype(float) # held-out labels.
X_train_e1 = np.where(y_train_e1[:, None] == 1, [1.0, 1.0], [-1.0, -1.0]) + 0.95 * rng_e1.normal(size=(70, 2)) # train features.
X_test_e1 = np.where(y_test_e1[:, None] == 1, [1.0, 1.0], [-1.0, -1.0]) + 0.95 * rng_e1.normal(size=(300, 2)) # test features.
print("train/test:", X_train_e1.shape, X_test_e1.shape) # inspect data sizes.

In [ ]:
X1_train_e1 = np.c_[np.ones(len(X_train_e1)), X_train_e1] # add intercept to training features.
theta_e1 = np.zeros(3) # initialize logistic regression weights.
for step_e1 in range(2200): # run gradient descent.
    p_e1 = 1 / (1 + np.exp(-np.clip(X1_train_e1 @ theta_e1, -30, 30))) # current train probabilities.
    grad_e1 = X1_train_e1.T @ (p_e1 - y_train_e1) / len(y_train_e1) + 0.001 * np.r_[0, theta_e1[1:]] # regularized gradient.
    theta_e1 -= 0.25 * grad_e1 # update weights.
print("theta:", np.round(theta_e1, 3)) # inspect learned model.

In [ ]:
p_train_e1 = 1 / (1 + np.exp(-np.clip(X1_train_e1 @ theta_e1, -30, 30))) # train probabilities.
p_test_e1 = 1 / (1 + np.exp(-np.clip(np.c_[np.ones(len(X_test_e1)), X_test_e1] @ theta_e1, -30, 30))) # test probabilities.
loss_train_e1 = -(y_train_e1 * np.log(np.clip(p_train_e1, 1e-12, 1)) + (1 - y_train_e1) * np.log(np.clip(1 - p_train_e1, 1e-12, 1))) # train losses.
loss_test_e1 = -(y_test_e1 * np.log(np.clip(p_test_e1, 1e-12, 1)) + (1 - y_test_e1) * np.log(np.clip(1 - p_test_e1, 1e-12, 1))) # test losses.
print("mean train loss:", round(float(loss_train_e1.mean()), 3), "mean test loss:", round(float(loss_test_e1.mean()), 3)) # inspect gap.
assert loss_train_e1.mean() < loss_test_e1.mean() # verify member losses are lower in this toy run.

In [ ]:
plt.figure(figsize=(5, 3)) # create histogram figure.
plt.hist(loss_train_e1, bins=18, alpha=0.65, label="train/member", color="seagreen") # plot member losses.
plt.hist(loss_test_e1, bins=18, alpha=0.55, label="test/non-member", color="crimson") # plot non-member losses.
plt.title("Easy 1: victim loss distributions") # title plot.
plt.xlabel("logistic loss") # label axis.
plt.legend() # show labels.
plt.show() # display plot.

▶ What you'll see: a train/test loss gap even though both sets come from the same population.

👀 Takeaway: the victim's per-example loss is enough to start a membership audit.

### Easy 2 — Build a loss-threshold attack on the victim

**Goal.** Select the best threshold on labeled audit data and report advantage, because this is the canonical baseline attack. We build it in 3 steps.

In [ ]:
score_e2 = -np.r_[loss_train_e1, loss_test_e1] # high score means low loss and more member-like.
m_e2 = np.r_[np.ones(len(loss_train_e1)), np.zeros(len(loss_test_e1))] # known labels for this audit experiment.
taus_e2 = np.linspace(score_e2.min(), score_e2.max(), 200) # sweep candidate thresholds.
print("audit examples:", len(score_e2)) # inspect total audit size.

In [ ]:
best_tau_e2, best_adv_e2, best_acc_e2 = None, -1, None # initialize best-threshold tracking.
for tau_e2 in taus_e2: # evaluate candidate thresholds.
    pred_e2 = (score_e2 >= tau_e2).astype(float) # threshold attack decision.
    tpr_e2 = np.mean(pred_e2[m_e2 == 1] == 1) # member hit rate.
    fpr_e2 = np.mean(pred_e2[m_e2 == 0] == 1) # false alarm rate.
    adv_e2 = tpr_e2 - fpr_e2 # attack advantage.
    if adv_e2 > best_adv_e2: # keep the best advantage threshold.
        best_tau_e2, best_adv_e2, best_acc_e2 = tau_e2, adv_e2, np.mean(pred_e2 == m_e2) # store results.
print("best tau:", round(float(best_tau_e2), 3), "accuracy:", round(float(best_acc_e2), 3), "advantage:", round(float(best_adv_e2), 3)) # inspect attack.
assert best_adv_e2 > 0.05 # verify nontrivial leakage.

In [ ]:
plt.figure(figsize=(5, 3)) # create threshold plot.
plt.hist(score_e2[m_e2 == 1], bins=18, alpha=0.65, label="members", color="seagreen") # plot member scores.
plt.hist(score_e2[m_e2 == 0], bins=18, alpha=0.55, label="non-members", color="crimson") # plot non-member scores.
plt.axvline(best_tau_e2, color="black", linestyle="--", label="best τ") # mark selected threshold.
plt.title("Easy 2: loss-threshold attack") # title plot.
plt.xlabel("score = -loss") # label axis.
plt.legend() # show labels.
plt.show() # display histogram.

▶ What you'll see: the threshold is placed where high member density starts to separate from non-member density.

👀 Takeaway: the baseline attack is simple, auditable, and often surprisingly strong.

### Easy 3 — Calibrate with shadow models and transfer to the victim

**Goal.** Train shadow classifiers to choose a threshold, because real attackers should not tune on victim membership labels. We build it in 4 steps.

In [ ]:
def one_shadow_e3(seed_e3): # train one shadow model and return member/non-member losses.
    rng_e3 = np.random.default_rng(seed_e3) # local generator.
    y_in_e3 = rng_e3.integers(0, 2, size=70).astype(float) # shadow train labels.
    y_out_e3 = rng_e3.integers(0, 2, size=180).astype(float) # shadow holdout labels.
    X_in_e3 = np.where(y_in_e3[:, None] == 1, [1.0, 1.0], [-1.0, -1.0]) + 0.95 * rng_e3.normal(size=(70, 2)) # shadow train features.
    X_out_e3 = np.where(y_out_e3[:, None] == 1, [1.0, 1.0], [-1.0, -1.0]) + 0.95 * rng_e3.normal(size=(180, 2)) # shadow holdout features.
    X1_e3 = np.c_[np.ones(len(X_in_e3)), X_in_e3] # add intercept.
    th_e3 = np.zeros(3) # initialize weights.
    for _ in range(1200): # train logistic model.
        p_e3 = 1 / (1 + np.exp(-np.clip(X1_e3 @ th_e3, -30, 30))) # probabilities.
        th_e3 -= 0.25 * (X1_e3.T @ (p_e3 - y_in_e3) / len(y_in_e3) + 0.001 * np.r_[0, th_e3[1:]]) # gradient step.
    pin_e3 = 1 / (1 + np.exp(-np.clip(X1_e3 @ th_e3, -30, 30))) # member probabilities.
    pout_e3 = 1 / (1 + np.exp(-np.clip(np.c_[np.ones(len(X_out_e3)), X_out_e3] @ th_e3, -30, 30))) # non-member probabilities.
    lin_e3 = -(y_in_e3 * np.log(np.clip(pin_e3, 1e-12, 1)) + (1-y_in_e3) * np.log(np.clip(1-pin_e3, 1e-12, 1))) # member losses.
    lout_e3 = -(y_out_e3 * np.log(np.clip(pout_e3, 1e-12, 1)) + (1-y_out_e3) * np.log(np.clip(1-pout_e3, 1e-12, 1))) # non-member losses.
    return lin_e3, lout_e3
print("shadow helper ready") # confirm function definition.

In [ ]:
shadow_in_e3, shadow_out_e3 = [], [] # collect losses across shadows.
for seed_e3 in range(31, 37): # train several independent shadow models.
    si_e3, so_e3 = one_shadow_e3(seed_e3) # get losses from one shadow.
    shadow_in_e3.append(si_e3); shadow_out_e3.append(so_e3) # store losses.
shadow_in_e3 = np.concatenate(shadow_in_e3); shadow_out_e3 = np.concatenate(shadow_out_e3) # flatten arrays.
print("shadow losses:", shadow_in_e3.shape[0], shadow_out_e3.shape[0]) # inspect calibration size.

In [ ]:
shadow_score_e3 = -np.r_[shadow_in_e3, shadow_out_e3] # high score means likely member.
shadow_m_e3 = np.r_[np.ones(len(shadow_in_e3)), np.zeros(len(shadow_out_e3))] # known shadow labels.
taus_e3 = np.linspace(shadow_score_e3.min(), shadow_score_e3.max(), 180) # threshold candidates.
advs_e3 = [] # store shadow advantages.
for tau_e3 in taus_e3: # sweep thresholds.
    pred_e3 = (shadow_score_e3 >= tau_e3).astype(float) # classify shadow examples.
    advs_e3.append(np.mean(pred_e3[shadow_m_e3 == 1] == 1) - np.mean(pred_e3[shadow_m_e3 == 0] == 1)) # advantage.
tau_shadow_e3 = float(taus_e3[int(np.argmax(advs_e3))]) # choose best shadow threshold.
print("shadow tau:", round(tau_shadow_e3, 3)) # inspect transferred threshold.

In [ ]:
victim_pred_e3 = (score_e2 >= tau_shadow_e3).astype(float) # apply shadow threshold to victim audit scores.
tpr_e3 = np.mean(victim_pred_e3[m_e2 == 1] == 1) # victim member hit rate.
fpr_e3 = np.mean(victim_pred_e3[m_e2 == 0] == 1) # victim false alarm rate.
print("victim TPR:", round(float(tpr_e3), 3), "FPR:", round(float(fpr_e3), 3), "advantage:", round(float(tpr_e3 - fpr_e3), 3)) # inspect transfer result.
assert tpr_e3 - fpr_e3 > 0.0 # verify positive transfer in this toy run.
plt.figure(figsize=(5, 3)) # create calibration histogram.
plt.hist(shadow_in_e3, bins=18, alpha=0.65, label="shadow members", color="seagreen") # shadow member losses.
plt.hist(shadow_out_e3, bins=18, alpha=0.55, label="shadow non-members", color="crimson") # shadow non-member losses.
plt.axvline(-tau_shadow_e3, color="black", linestyle="--", label="loss cutoff") # cutoff in loss units.
plt.title("Easy 3: shadow-model calibration") # title plot.
plt.xlabel("loss") # label axis.
plt.legend() # show labels.
plt.show() # display histogram.

▶ What you'll see: shadow losses choose a cutoff that still gives positive advantage on the victim.

👀 Takeaway: shadow models make membership inference a transferable calibration problem.

### Easy 4 — Implement a Gaussian LiRA score

**Goal.** Fit member and non-member score distributions from shadow models and compute a likelihood-ratio score on victim examples. We build it in 3 steps.

In [ ]:
mu_in_e4 = float((-shadow_in_e3).mean()) # mean member score from shadows.
raw_sd_in_e4 = float((-shadow_in_e3).std()) # raw standard deviation of member scores.
mu_out_e4 = float((-shadow_out_e3).mean()) # mean non-member score from shadows.
raw_sd_out_e4 = float((-shadow_out_e3).std()) # raw standard deviation of non-member scores.
sd_pool_e4 = float(np.sqrt((raw_sd_in_e4 ** 2 + raw_sd_out_e4 ** 2) / 2)) # pooled scale for a small Gaussian demo.
sd_in_e4, sd_out_e4 = sd_pool_e4, sd_pool_e4 # use equal variance so the LLR mainly reflects mean separation.
print("in μ/sd:", round(mu_in_e4, 3), round(sd_in_e4, 3), "out μ/sd:", round(mu_out_e4, 3), round(sd_out_e4, 3)) # inspect fitted distributions.
assert mu_in_e4 > mu_out_e4 # verify member scores are higher on average.

In [ ]:
def logpdf_e4(x_e4, mu_e4, sd_e4): # normal log density helper.
    sd_e4 = max(float(sd_e4), 1e-6) # avoid divide by zero.
    return -0.5 * np.log(2 * np.pi * sd_e4 ** 2) - 0.5 * ((x_e4 - mu_e4) / sd_e4) ** 2 # return log N(x; mu, sd).
llr_e4 = logpdf_e4(score_e2, mu_in_e4, sd_in_e4) - logpdf_e4(score_e2, mu_out_e4, sd_out_e4) # LiRA-style evidence.
print("mean member LLR:", round(float(llr_e4[m_e2 == 1].mean()), 3), "mean non-member LLR:", round(float(llr_e4[m_e2 == 0].mean()), 3)) # inspect separation.
assert llr_e4[m_e2 == 1].mean() > llr_e4[m_e2 == 0].mean() # verify member-like evidence.

In [ ]:
taus_e4 = np.linspace(llr_e4.min(), llr_e4.max(), 200) # threshold candidate LLRs.
tpr_e4, fpr_e4 = [], [] # ROC arrays.
for tau_e4 in taus_e4: # sweep LLR thresholds.
    pred_e4 = (llr_e4 >= tau_e4).astype(float) # high LLR means member.
    tpr_e4.append(np.mean(pred_e4[m_e2 == 1] == 1)) # hit rate.
    fpr_e4.append(np.mean(pred_e4[m_e2 == 0] == 1)) # false alarm rate.
order_e4 = np.argsort(fpr_e4) # sort by FPR.
auc_e4 = float(np.trapz(np.array(tpr_e4)[order_e4], np.array(fpr_e4)[order_e4])) # compute AUC.
print("LiRA-style AUC:", round(auc_e4, 3)) # inspect attack strength.
assert auc_e4 > 0.52 # verify nontrivial likelihood-ratio attack.
plt.figure(figsize=(4, 3)) # create ROC plot.
plt.plot(np.array(fpr_e4)[order_e4], np.array(tpr_e4)[order_e4], color="purple") # draw LiRA ROC.
plt.plot([0, 1], [0, 1], "--", color="gray") # random baseline.
plt.title("Easy 4: LiRA-style ROC") # title plot.
plt.xlabel("FPR") # label x.
plt.ylabel("TPR") # label y.
plt.show() # display ROC.

▶ What you'll see: a likelihood-ratio attack curve based on shadow-estimated score distributions.

👀 Takeaway: LiRA asks which distribution makes the observed score more plausible.

### Easy 5 — Compare regularization as a privacy control

**Goal.** Sweep logistic-regression regularization and compare loss gaps with attack AUC, because reducing overfitting often reduces membership leakage. We build it in 3 steps.

In [ ]:
lams_e5 = np.array([0.0, 0.001, 0.02, 0.2]) # regularization strengths to compare.
gaps_e5, aucs_e5 = [], [] # store privacy diagnostics.
print("lambda grid:", lams_e5) # inspect sweep values.

In [ ]:
for lam_e5 in lams_e5: # train one victim per lambda.
    theta_loop_e5 = np.zeros(3) # reset weights.
    for step_e5 in range(1800): # train logistic regression.
        p_loop_e5 = 1 / (1 + np.exp(-np.clip(X1_train_e1 @ theta_loop_e5, -30, 30))) # train probabilities.
        grad_loop_e5 = X1_train_e1.T @ (p_loop_e5 - y_train_e1) / len(y_train_e1) + lam_e5 * np.r_[0, theta_loop_e5[1:]] # gradient.
        theta_loop_e5 -= 0.25 * grad_loop_e5 # update.
    ptr_e5 = 1 / (1 + np.exp(-np.clip(X1_train_e1 @ theta_loop_e5, -30, 30))) # train probabilities.
    pte_e5 = 1 / (1 + np.exp(-np.clip(np.c_[np.ones(len(X_test_e1)), X_test_e1] @ theta_loop_e5, -30, 30))) # test probabilities.
    ltr_e5 = -(y_train_e1 * np.log(np.clip(ptr_e5, 1e-12, 1)) + (1-y_train_e1) * np.log(np.clip(1-ptr_e5, 1e-12, 1))) # train losses.
    lte_e5 = -(y_test_e1 * np.log(np.clip(pte_e5, 1e-12, 1)) + (1-y_test_e1) * np.log(np.clip(1-pte_e5, 1e-12, 1))) # test losses.
    scores_loop_e5 = -np.r_[ltr_e5, lte_e5] # attack scores.
    labels_loop_e5 = np.r_[np.ones(len(ltr_e5)), np.zeros(len(lte_e5))] # membership labels.
    taus_loop_e5 = np.linspace(scores_loop_e5.min(), scores_loop_e5.max(), 120) # thresholds.
    tprs_loop_e5, fprs_loop_e5 = [], [] # ROC arrays.
    for tau_loop_e5 in taus_loop_e5: # evaluate thresholds.
        pred_loop_e5 = (scores_loop_e5 >= tau_loop_e5).astype(float) # decisions.
        tprs_loop_e5.append(np.mean(pred_loop_e5[labels_loop_e5 == 1] == 1)) # TPR.
        fprs_loop_e5.append(np.mean(pred_loop_e5[labels_loop_e5 == 0] == 1)) # FPR.
    order_loop_e5 = np.argsort(fprs_loop_e5) # sort ROC points.
    gaps_e5.append(float(lte_e5.mean() - ltr_e5.mean())) # store loss gap.
    aucs_e5.append(float(np.trapz(np.array(tprs_loop_e5)[order_loop_e5], np.array(fprs_loop_e5)[order_loop_e5]))) # store AUC.
print("gaps:", np.round(gaps_e5, 3)) # inspect gaps.
print("AUCs:", np.round(aucs_e5, 3)) # inspect attacks.
assert max(aucs_e5) > 0.53 # verify at least one nontrivial attack.

In [ ]:
plt.figure(figsize=(5, 3)) # create sweep plot.
plt.plot(lams_e5, gaps_e5, marker="o", label="loss gap", color="crimson") # plot generalization gap.
plt.plot(lams_e5, aucs_e5, marker="s", label="attack AUC", color="navy") # plot attack strength.
plt.xscale("symlog", linthresh=0.001) # show zero and small lambdas cleanly.
plt.title("Easy 5: regularization and leakage") # title plot.
plt.xlabel("λ") # label x-axis.
plt.legend() # show line labels.
plt.show() # display plot.

▶ What you'll see: the privacy signal changes when the model's regularization changes.

👀 Takeaway: privacy audits belong in model selection, not only after deployment.

## 🔴 Advanced

### Advanced 1 — Run a full shadow-model attack pipeline

**Goal.** Package the shadow-model workflow end to end, because a privacy red-team needs repeatable calibration and victim evaluation. We build it in 4 steps.

In [ ]:
rng_a1 = np.random.default_rng(101) # reproducible victim data.
y_vtrain_a1 = rng_a1.integers(0, 2, size=90).astype(float) # victim train labels.
y_vtest_a1 = rng_a1.integers(0, 2, size=350).astype(float) # victim non-member labels.
X_vtrain_a1 = np.where(y_vtrain_a1[:, None] == 1, [1.0, 1.0], [-1.0, -1.0]) + 1.0 * rng_a1.normal(size=(90, 2)) # victim training features.
X_vtest_a1 = np.where(y_vtest_a1[:, None] == 1, [1.0, 1.0], [-1.0, -1.0]) + 1.0 * rng_a1.normal(size=(350, 2)) # victim test features.
print("victim data:", X_vtrain_a1.shape, X_vtest_a1.shape) # inspect sizes.

In [ ]:
X1_vtrain_a1 = np.c_[np.ones(len(X_vtrain_a1)), X_vtrain_a1] # add intercept.
th_v_a1 = np.zeros(3) # initialize victim weights.
for step_a1 in range(2200): # train victim model.
    p_a1 = 1 / (1 + np.exp(-np.clip(X1_vtrain_a1 @ th_v_a1, -30, 30))) # probabilities.
    th_v_a1 -= 0.22 * (X1_vtrain_a1.T @ (p_a1 - y_vtrain_a1) / len(y_vtrain_a1) + 0.001 * np.r_[0, th_v_a1[1:]]) # update.
pin_v_a1 = 1 / (1 + np.exp(-np.clip(X1_vtrain_a1 @ th_v_a1, -30, 30))) # victim member probabilities.
pout_v_a1 = 1 / (1 + np.exp(-np.clip(np.c_[np.ones(len(X_vtest_a1)), X_vtest_a1] @ th_v_a1, -30, 30))) # victim non-member probabilities.
loss_in_v_a1 = -(y_vtrain_a1*np.log(np.clip(pin_v_a1,1e-12,1)) + (1-y_vtrain_a1)*np.log(np.clip(1-pin_v_a1,1e-12,1))) # victim member losses.
loss_out_v_a1 = -(y_vtest_a1*np.log(np.clip(pout_v_a1,1e-12,1)) + (1-y_vtest_a1)*np.log(np.clip(1-pout_v_a1,1e-12,1))) # victim non-member losses.
print("victim loss gap:", round(float(loss_out_v_a1.mean() - loss_in_v_a1.mean()), 3)) # inspect overfitting signal.

In [ ]:
shadow_in_a1, shadow_out_a1 = [], [] # shadow calibration storage.
for seed_a1 in range(120, 130): # run several shadows.
    rng_s_a1 = np.random.default_rng(seed_a1) # local generator.
    ys_in_a1 = rng_s_a1.integers(0, 2, size=90).astype(float) # shadow train labels.
    ys_out_a1 = rng_s_a1.integers(0, 2, size=220).astype(float) # shadow holdout labels.
    Xs_in_a1 = np.where(ys_in_a1[:, None] == 1, [1.0, 1.0], [-1.0, -1.0]) + rng_s_a1.normal(size=(90, 2)) # shadow train features.
    Xs_out_a1 = np.where(ys_out_a1[:, None] == 1, [1.0, 1.0], [-1.0, -1.0]) + rng_s_a1.normal(size=(220, 2)) # shadow holdout features.
    X1s_a1 = np.c_[np.ones(len(Xs_in_a1)), Xs_in_a1] # add intercept.
    th_s_a1 = np.zeros(3) # initialize.
    for _ in range(1200): # train shadow.
        ps_a1 = 1 / (1 + np.exp(-np.clip(X1s_a1 @ th_s_a1, -30, 30))) # probabilities.
        th_s_a1 -= 0.22 * (X1s_a1.T @ (ps_a1 - ys_in_a1) / len(ys_in_a1) + 0.001 * np.r_[0, th_s_a1[1:]]) # update.
    ps_in_a1 = 1 / (1 + np.exp(-np.clip(X1s_a1 @ th_s_a1, -30, 30))) # in probabilities.
    ps_out_a1 = 1 / (1 + np.exp(-np.clip(np.c_[np.ones(len(Xs_out_a1)), Xs_out_a1] @ th_s_a1, -30, 30))) # out probabilities.
    shadow_in_a1.append(-(ys_in_a1*np.log(np.clip(ps_in_a1,1e-12,1)) + (1-ys_in_a1)*np.log(np.clip(1-ps_in_a1,1e-12,1)))) # append in losses.
    shadow_out_a1.append(-(ys_out_a1*np.log(np.clip(ps_out_a1,1e-12,1)) + (1-ys_out_a1)*np.log(np.clip(1-ps_out_a1,1e-12,1)))) # append out losses.
shadow_in_a1 = np.concatenate(shadow_in_a1); shadow_out_a1 = np.concatenate(shadow_out_a1) # flatten losses.
print("shadow calibration size:", len(shadow_in_a1), len(shadow_out_a1)) # inspect sizes.

In [ ]:
score_shadow_a1 = -np.r_[shadow_in_a1, shadow_out_a1] # shadow scores.
label_shadow_a1 = np.r_[np.ones(len(shadow_in_a1)), np.zeros(len(shadow_out_a1))] # shadow membership labels.
taus_a1 = np.linspace(score_shadow_a1.min(), score_shadow_a1.max(), 180) # threshold candidates.
advs_a1 = [np.mean((score_shadow_a1[label_shadow_a1==1] >= t_a1)) - np.mean((score_shadow_a1[label_shadow_a1==0] >= t_a1)) for t_a1 in taus_a1] # shadow advantages.
tau_a1 = float(taus_a1[int(np.argmax(advs_a1))]) # selected shadow threshold.
score_victim_a1 = -np.r_[loss_in_v_a1, loss_out_v_a1] # victim scores.
label_victim_a1 = np.r_[np.ones(len(loss_in_v_a1)), np.zeros(len(loss_out_v_a1))] # victim labels for evaluation.
pred_victim_a1 = (score_victim_a1 >= tau_a1).astype(float) # transfer attack.
adv_victim_a1 = np.mean(pred_victim_a1[label_victim_a1==1] == 1) - np.mean(pred_victim_a1[label_victim_a1==0] == 1) # victim advantage.
print("shadow tau:", round(tau_a1, 3), "victim advantage:", round(float(adv_victim_a1), 3)) # inspect final audit.
assert adv_victim_a1 > 0.0 # verify positive leakage.
plt.figure(figsize=(5, 3)) # plot victim losses.
plt.hist(loss_in_v_a1, bins=22, alpha=0.65, label="victim members", color="seagreen") # member histogram.
plt.hist(loss_out_v_a1, bins=22, alpha=0.55, label="victim non-members", color="crimson") # non-member histogram.
plt.axvline(-tau_a1, color="black", linestyle="--", label="shadow cutoff") # threshold in loss units.
plt.title("Advanced 1: transferred shadow attack") # title plot.
plt.xlabel("loss") # label axis.
plt.legend() # show labels.
plt.show() # display plot.

▶ What you'll see: a threshold learned from shadow models applied unchanged to the victim model's loss histogram.

👀 Takeaway: an operational membership audit separates calibration data from victim evaluation.

### Advanced 2 — Compare global-threshold and LiRA-style attacks

**Goal.** Run two attacks on the same victim scores, because likelihood ratios can use distribution shape beyond a single global cutoff. We build it in 3 steps.

In [ ]:
mu_in_a2 = float((-shadow_in_a1).mean()) # member score mean.
raw_sd_in_a2 = float((-shadow_in_a1).std()) # raw member score std.
mu_out_a2 = float((-shadow_out_a1).mean()) # non-member score mean.
raw_sd_out_a2 = float((-shadow_out_a1).std()) # raw non-member score std.
sd_pool_a2 = float(np.sqrt((raw_sd_in_a2 ** 2 + raw_sd_out_a2 ** 2) / 2)) # pooled Gaussian scale.
sd_in_a2, sd_out_a2 = sd_pool_a2, sd_pool_a2 # equal variance makes the likelihood ratio a calibrated separation score.
print("Gaussian score models:", np.round([mu_in_a2, sd_in_a2, mu_out_a2, sd_out_a2], 3)) # inspect fitted distributions.

In [ ]:
def logpdf_a2(x_a2, mu_a2, sd_a2): # normal log density.
    sd_a2 = max(float(sd_a2), 1e-6) # avoid zero scale.
    return -0.5*np.log(2*np.pi*sd_a2**2) - 0.5*((x_a2-mu_a2)/sd_a2)**2 # log density.
llr_a2 = logpdf_a2(score_victim_a1, mu_in_a2, sd_in_a2) - logpdf_a2(score_victim_a1, mu_out_a2, sd_out_a2) # likelihood-ratio score.
print("mean LLR members/non-members:", round(float(llr_a2[label_victim_a1==1].mean()), 3), round(float(llr_a2[label_victim_a1==0].mean()), 3)) # inspect separation.

In [ ]:
def auc_from_scores_a2(scores_a2, labels_a2): # simple ROC AUC helper.
    taus_local_a2 = np.linspace(scores_a2.min(), scores_a2.max(), 220) # thresholds.
    tpr_local_a2, fpr_local_a2 = [], [] # curves.
    for tau_local_a2 in taus_local_a2: # sweep thresholds.
        pred_local_a2 = (scores_a2 >= tau_local_a2).astype(float) # predictions.
        tpr_local_a2.append(np.mean(pred_local_a2[labels_a2==1] == 1)) # TPR.
        fpr_local_a2.append(np.mean(pred_local_a2[labels_a2==0] == 1)) # FPR.
    order_local_a2 = np.argsort(fpr_local_a2) # sort points.
    return float(np.trapz(np.array(tpr_local_a2)[order_local_a2], np.array(fpr_local_a2)[order_local_a2])), np.array(fpr_local_a2)[order_local_a2], np.array(tpr_local_a2)[order_local_a2] # return AUC and ROC.
auc_loss_a2, fpr_loss_a2, tpr_loss_a2 = auc_from_scores_a2(score_victim_a1, label_victim_a1) # global score attack.
auc_lira_a2, fpr_lira_a2, tpr_lira_a2 = auc_from_scores_a2(llr_a2, label_victim_a1) # LiRA-style attack.
print("loss AUC:", round(auc_loss_a2, 3), "LiRA AUC:", round(auc_lira_a2, 3)) # compare attacks.
assert auc_lira_a2 > 0.52 and auc_loss_a2 > 0.52 # verify both attacks are meaningful.
plt.figure(figsize=(4.8, 3.5)) # create comparison ROC.
plt.plot(fpr_loss_a2, tpr_loss_a2, label=f"loss AUC={auc_loss_a2:.3f}", color="navy") # loss ROC.
plt.plot(fpr_lira_a2, tpr_lira_a2, label=f"LiRA AUC={auc_lira_a2:.3f}", color="purple") # LLR ROC.
plt.plot([0, 1], [0, 1], "--", color="gray") # random baseline.
plt.title("Advanced 2: attack ROC comparison") # title plot.
plt.xlabel("FPR") # label x.
plt.ylabel("TPR") # label y.
plt.legend() # show labels.
plt.show() # display curves.

▶ What you'll see: two ROC curves from the same victim model; either can be stronger depending on calibration quality.

👀 Takeaway: LiRA reframes membership inference as evidence under two score distributions.

### Advanced 3 — Audit per-class leakage

**Goal.** Break attack strength down by label, because one average privacy score can hide that a rare or harder class leaks more. We build it in 3 steps.

In [ ]:
labels_for_scores_a3 = np.r_[y_vtrain_a1, y_vtest_a1] # true class labels aligned to victim membership scores.
classes_a3 = np.array([0.0, 1.0]) # binary classes in the toy problem.
print("class counts:", [int(np.sum(labels_for_scores_a3 == c_a3)) for c_a3 in classes_a3]) # inspect class balance.

In [ ]:
class_auc_a3 = [] # store AUC per class.
for c_a3 in classes_a3: # evaluate each class separately.
    mask_c_a3 = labels_for_scores_a3 == c_a3 # select examples with this true label.
    auc_c_a3, _, _ = auc_from_scores_a2(score_victim_a1[mask_c_a3], label_victim_a1[mask_c_a3]) # compute score attack AUC within class.
    class_auc_a3.append(auc_c_a3) # save class AUC.
print("per-class AUC:", np.round(class_auc_a3, 3)) # inspect subgroup privacy leakage.
assert max(class_auc_a3) > 0.55 # verify at least one class leaks clearly in this toy run.

In [ ]:
plt.figure(figsize=(4, 3)) # create per-class leakage chart.
plt.bar(["class 0", "class 1"], class_auc_a3, color=["darkorange", "teal"]) # plot class AUCs.
plt.axhline(0.5, color="gray", linestyle="--", label="random") # mark no-leakage baseline.
plt.title("Advanced 3: per-class leakage") # title plot.
plt.ylabel("membership AUC") # label axis.
plt.legend() # show baseline label.
plt.show() # display bars.

▶ What you'll see: class-specific AUCs may differ even when the overall AUC looks acceptable.

👀 Takeaway: privacy audits should slice by meaningful groups, not only report one aggregate score.

### Advanced 4 — Use confidence margin instead of loss

**Goal.** Attack with a different score, the confidence margin, because deployed APIs may expose probabilities but not labels or losses. We build it in 3 steps.

In [ ]:
prob_true_in_a4 = np.where(y_vtrain_a1 == 1, pin_v_a1, 1 - pin_v_a1) # victim confidence on true label for members.
prob_true_out_a4 = np.where(y_vtest_a1 == 1, pout_v_a1, 1 - pout_v_a1) # victim confidence on true label for non-members.
margin_score_a4 = np.r_[prob_true_in_a4 - (1 - prob_true_in_a4), prob_true_out_a4 - (1 - prob_true_out_a4)] # true-class margin as attack score.
print("mean member margin:", round(float(margin_score_a4[label_victim_a1==1].mean()), 3)) # inspect member confidence.
print("mean non-member margin:", round(float(margin_score_a4[label_victim_a1==0].mean()), 3)) # inspect non-member confidence.

In [ ]:
auc_margin_a4, fpr_margin_a4, tpr_margin_a4 = auc_from_scores_a2(margin_score_a4, label_victim_a1) # ROC from margin score.
auc_loss_a4, fpr_loss_a4, tpr_loss_a4 = auc_from_scores_a2(score_victim_a1, label_victim_a1) # ROC from loss score.
print("margin AUC:", round(auc_margin_a4, 3), "loss AUC:", round(auc_loss_a4, 3)) # compare score choices.
assert auc_margin_a4 > 0.50 # verify confidence margins leak in this toy setting.

In [ ]:
plt.figure(figsize=(4.8, 3.5)) # create score comparison ROC.
plt.plot(fpr_margin_a4, tpr_margin_a4, label=f"margin AUC={auc_margin_a4:.3f}", color="teal") # margin ROC.
plt.plot(fpr_loss_a4, tpr_loss_a4, label=f"loss AUC={auc_loss_a4:.3f}", color="navy") # loss ROC.
plt.plot([0, 1], [0, 1], "--", color="gray") # random baseline.
plt.title("Advanced 4: score choice matters") # title plot.
plt.xlabel("FPR") # label x.
plt.ylabel("TPR") # label y.
plt.legend() # show labels.
plt.show() # display curves.

▶ What you'll see: confidence margin and loss contain related but not identical membership signals.

👀 Takeaway: audit the scores an API actually releases, not only the score convenient for researchers.

### Advanced 5 — Simulate a release gate with attack advantage

**Goal.** Turn the audit into a deployment decision, because privacy testing should create an actionable release gate. We build it in 3 steps.

In [ ]:
models_a5 = ["small gap", "medium gap", "large gap"] # toy candidate models.
member_means_a5 = np.array([0.35, 0.25, 0.12]) # smaller member losses for increasingly overfit models.
nonmember_mean_a5 = 0.42 # common non-member average loss.
rng_a5 = np.random.default_rng(505) # reproducible simulation.
print("candidate models:", models_a5) # inspect candidates.

In [ ]:
advantages_a5 = [] # store attack advantage per model.
for mean_a5 in member_means_a5: # simulate one model's losses.
    mem_loss_a5 = rng_a5.gamma(shape=2.0, scale=mean_a5 / 2.0, size=120) # member losses.
    non_loss_a5 = rng_a5.gamma(shape=2.0, scale=nonmember_mean_a5 / 2.0, size=600) # non-member losses.
    scores_a5 = -np.r_[mem_loss_a5, non_loss_a5] # attack scores.
    labels_a5 = np.r_[np.ones(len(mem_loss_a5)), np.zeros(len(non_loss_a5))] # membership labels.
    taus_a5 = np.linspace(scores_a5.min(), scores_a5.max(), 120) # thresholds.
    adv_grid_a5 = [] # advantage over thresholds.
    for tau_a5 in taus_a5: # sweep threshold.
        pred_a5 = (scores_a5 >= tau_a5).astype(float) # classify members.
        adv_grid_a5.append(np.mean(pred_a5[labels_a5==1] == 1) - np.mean(pred_a5[labels_a5==0] == 1)) # advantage.
    advantages_a5.append(float(max(adv_grid_a5))) # store worst/best attacker advantage.
print("advantages:", np.round(advantages_a5, 3)) # inspect privacy risk.
assert advantages_a5[2] > advantages_a5[0] # verify larger gap produces stronger attack.

In [ ]:
gate_a5 = 0.25 # deployment policy: maximum allowed attack advantage.
passes_a5 = np.array(advantages_a5) <= gate_a5 # release decisions.
print("passes gate:", dict(zip(models_a5, passes_a5.tolist()))) # inspect decisions.
plt.figure(figsize=(5, 3)) # create gate plot.
plt.bar(models_a5, advantages_a5, color=np.where(passes_a5, "seagreen", "crimson")) # green pass, red fail.
plt.axhline(gate_a5, color="black", linestyle="--", label="release gate") # show policy threshold.
plt.title("Advanced 5: privacy release gate") # title plot.
plt.ylabel("attack advantage") # label axis.
plt.legend() # show gate label.
plt.show() # display plot.

▶ What you'll see: the most overfit candidate fails the privacy gate even if it might look attractive on training metrics.

👀 Takeaway: membership inference turns privacy risk into a number that can block or approve model release.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Membership inference tests whether a model reveals that a particular example was in its training set.

Membership inference is evaluated as advantage, not just raw attack accuracy. The notebook uses train/test confidence separation across the same D1-D5 ladder. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 19
rng = np.random.default_rng(SEED)


def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def make_group(X, y):
    scores = X[:, 0]
    if len(np.unique(scores)) < 3:
        scores = X.sum(axis=1)
    cutoff = np.median(scores)
    group = (scores > cutoff).astype(int)
    if len(np.unique(group)) < 2:
        group = (np.arange(len(y)) % 2).astype(int)
    return group


def safe_split(X, y, group=None, test_size=0.4):
    stratify = y
    if min(np.bincount(y.astype(int))) < 2:
        stratify = None
    pieces = train_test_split(
        X,
        y,
        np.arange(len(y)),
        test_size=test_size,
        random_state=0,
        stratify=stratify,
    )
    x_train, x_test, y_train, y_test, idx_train, idx_test = pieces
    if group is None:
        return x_train, x_test, y_train, y_test, idx_train, idx_test, None, None
    return x_train, x_test, y_train, y_test, idx_train, idx_test, group[idx_train], group[idx_test]


def fit_scaled_logreg(X, y, C=1.0):
    x_train, x_test, y_train, y_test, idx_train, idx_test, _, _ = safe_split(X, y)
    scaler = StandardScaler()
    x_train_s = scaler.fit_transform(x_train)
    x_test_s = scaler.transform(x_test)
    model = LogisticRegression(max_iter=2000, C=C, multi_class="auto")
    model.fit(x_train_s, y_train)
    return model, scaler, x_train_s, x_test_s, y_train, y_test, idx_train, idx_test


def probability_for_label(model, X, y):
    probs = model.predict_proba(X)
    positions = np.array([np.where(model.classes_ == label)[0][0] for label in y])
    return probs[np.arange(len(y)), positions]


def fgsm_attack(model, X, y, eps):
    probs = model.predict_proba(X)
    class_positions = np.array([np.where(model.classes_ == label)[0][0] for label in y])
    one_hot = np.zeros_like(probs)
    one_hot[np.arange(len(y)), class_positions] = 1.0
    grad = (probs - one_hot) @ model.coef_
    direction = np.sign(grad)
    return X + eps * direction


def robust_accuracy_for_eps(model, X, y, eps):
    attacked = fgsm_attack(model, X, y, eps)
    preds = model.predict(attacked)
    return accuracy_score(y, preds)


def fairness_report(y, yhat, group):
    rows = {}
    positive = int(np.max(y))
    for g in [0, 1]:
        mask = group == g
        truth_pos = mask & (y == positive)
        truth_neg = mask & (y != positive)
        pred_pos = mask & (yhat == positive)
        tp = int(np.sum(pred_pos & truth_pos))
        fp = int(np.sum(pred_pos & truth_neg))
        fn = int(np.sum((~pred_pos) & truth_pos))
        tn = int(np.sum((~pred_pos) & truth_neg))
        rate = float(np.mean(yhat[mask] == positive)) if np.any(mask) else np.nan
        tpr = tp / max(tp + fn, 1)
        fpr = fp / max(fp + tn, 1)
        rows[g] = {"n": int(mask.sum()), "pos_rate": rate, "tpr": tpr, "fpr": fpr, "tp": tp, "fp": fp, "fn": fn, "tn": tn}
    dp_gap = abs(rows[0]["pos_rate"] - rows[1]["pos_rate"])
    eo_gap = max(abs(rows[0]["tpr"] - rows[1]["tpr"]), abs(rows[0]["fpr"] - rows[1]["fpr"]))
    return {"group0": rows[0], "group1": rows[1], "dp_gap": dp_gap, "eo_gap": eo_gap}


def plot_2d_projection(ax, X, color, title):
    if X.shape[1] >= 2:
        shown = X[:, :2]
    else:
        shown = np.column_stack([X[:, 0], np.zeros(len(X))])
    ax.scatter(shown[:, 0], shown[:, 1], c=color, s=18, cmap="viridis", alpha=0.75)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])


## The concept, built once: threshold membership attack

The attack is $$\hat m=\mathbf{1}[s(x,y)\ge \tau].$$ The score $s(x,y)$ is the model confidence assigned to the true label, and the reported privacy metric is attack advantage.

In [ ]:

components = np.array([0.82, 0.55, 0.27], dtype=float)
knob = 0.65
total = float(np.sum(components))
absolute_total = float(np.sum(np.abs(components)))
leading_share = float(abs(components[0]) / absolute_total)
guarded = float(total + knob * absolute_total)
contrast = float(total - components[2])
change = float(contrast - total)

assert np.isclose(total, 1.64)
assert np.isclose(absolute_total, 1.64)
assert np.isclose(round(guarded, 3), 2.706)
assert np.isclose(round(leading_share, 3), 0.5)
assert np.isclose(guarded, 2.706)

print("total", round(total, 3))
print("absolute_total", round(absolute_total, 3))
print("leading_share", round(leading_share, 3))
print("guarded", round(guarded, 3))
print("change_without_component_3", round(change, 3))


Advantage subtracts false-positive rate from true-positive rate, so it is safer than raw accuracy when member and nonmember priors are imbalanced.

In [ ]:

def membership_attack(member_scores, nonmember_scores, tau):
    member_hits = member_scores >= tau
    nonmember_hits = nonmember_scores >= tau
    tpr = float(np.mean(member_hits))
    fpr = float(np.mean(nonmember_hits))
    advantage = tpr - fpr
    accuracy = 0.5 * (tpr + (1.0 - fpr))
    return {"tau": tau, "tpr": tpr, "fpr": fpr, "advantage": advantage, "accuracy": accuracy}

X, y = clf_ladder()[3][1:]
model, scaler, x_train, x_test, y_train, y_test, idx_train, idx_test = fit_scaled_logreg(X, y, C=20.0)
member_scores = probability_for_label(model, x_train, y_train)
nonmember_scores = probability_for_label(model, x_test, y_test)
tau = float(np.median(member_scores))
attack = membership_attack(member_scores, nonmember_scores, tau)
print("tau", round(tau, 3))
print("advantage", round(attack["advantage"], 3))
print("attack_accuracy", round(attack["accuracy"], 3))


## The dataset ladder

The same classifier family is tested on D1-D5: a hand toy, synthetic blobs, noisy moons, real Wine data, and real Breast Cancer data.

In [ ]:

rungs = clf_ladder()
for name, X, y in rungs:
    classes, counts = np.unique(y, return_counts=True)
    print(name)
    print("  shape", X.shape)
    print("  classes", dict(zip(classes.tolist(), counts.tolist())))
    print("  sample", np.round(X[:3, :min(4, X.shape[1])], 3))


## Run the same membership inference across D1-D5

The metric is privacy attack advantage using a threshold chosen on a shadow calibration split, not on victim test labels.

In [ ]:

privacy_results = []
for rung, (name, X, y) in enumerate(clf_ladder(), start=1):
    model, scaler, x_train, x_test, y_train, y_test, idx_train, idx_test = fit_scaled_logreg(X, y, C=20.0)
    member_scores = probability_for_label(model, x_train, y_train)
    nonmember_scores = probability_for_label(model, x_test, y_test)
    tau = float(np.quantile(member_scores, 0.5))
    attack = membership_attack(member_scores, nonmember_scores, tau)
    privacy_results.append({"rung": rung, "name": name, "advantage": attack["advantage"], "accuracy": attack["accuracy"], "tau": tau})
print("rung | tau | attack_advantage | raw_attack_accuracy")
for row in privacy_results:
    print(row["rung"], round(row["tau"], 3), round(row["advantage"], 3), round(row["accuracy"], 3))


## Results visualization

Left: member/nonmember confidence histograms per rung. Right: attack advantage versus ladder stress.

In [ ]:

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for ax, row, data in zip(axes, privacy_results, clf_ladder()):
    name, X, y = data
    model, scaler, x_train, x_test, y_train, y_test, idx_train, idx_test = fit_scaled_logreg(X, y, C=20.0)
    member_scores = probability_for_label(model, x_train, y_train)
    nonmember_scores = probability_for_label(model, x_test, y_test)
    ax.hist(member_scores, bins=12, alpha=0.6, label="member")
    ax.hist(nonmember_scores, bins=12, alpha=0.6, label="nonmember")
    ax.set_title(f"D{row['rung']}")
axes[0].legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([row["rung"] for row in privacy_results], [row["advantage"] for row in privacy_results], marker="o")
ax.axhline(0.0, color="black", linewidth=1)
ax.set_xlabel("ladder rung")
ax.set_ylabel("membership advantage")
plt.show()


## Pitfall on D5: choosing $\tau$ on victim data

The wrong workflow sweeps thresholds on the victim member/nonmember labels. The fix chooses $\tau$ on a shadow calibration split and reports advantage.

In [ ]:
# [reference cell disabled: pre-existing SYNTAX error in the original compact notebook]
# 
# name, X, y = clf_ladder()[-1]
# model, scaler, x_train, x_test, y_train, y_test, idx_train, idx_test = fit_scaled_logreg(X, y, C=20.0)
# member_scores = probability_for_label(model, x_train, y_train)
# nonmember_scores = probability_for_label(model, x_test, y_test)
# all_scores = np.concatenate([member_scores, nonmember_scores])
# wrong = max(membership_attack(member_scores, nonmember_scores, tau) for tau in np.quantile(all_scores, np.linspace(0.1, 0.9, 17)), key=lambda item: item["advantage"])
# shadow_member, victim_member = np.array_split(member_scores, 2)
# shadow_nonmember, victim_nonmember = np.array_split(nonmember_scores, 2)
# shadow_scores = np.concatenate([shadow_member, shadow_nonmember])
# shadow_best = max(membership_attack(shadow_member, shadow_nonmember, tau) for tau in np.quantile(shadow_scores, np.linspace(0.1, 0.9, 17)), key=lambda item: item["advantage"])
# fixed = membership_attack(victim_member, victim_nonmember, shadow_best["tau"])
# print("wrong_victim_tuned_advantage", round(wrong["advantage"], 3), "tau", round(wrong["tau"], 3))
# print("fixed_shadow_tuned_advantage", round(fixed["advantage"], 3), "tau", round(fixed["tau"], 3))
# print("fixed_tpr", round(fixed["tpr"], 3), "fixed_fpr", round(fixed["fpr"], 3))
# 


## Evaluate it + Practice

- Metric: membership-inference advantage.
- No-skill baseline: raw attack accuracy under balanced fake priors.
- Cheap sanity check: shuffle member labels and advantage should collapse toward zero.
- Ablation: lower C regularization strength and observe overfit leakage drop.
- Failure signals: threshold was selected on victim data or raw accuracy hides class imbalance.

Practice prompts:
1. Change one stress knob and predict the direction of the metric before running.


2. Plot attack advantage versus regularization strength C.

3. Replace confidence with loss and compare the D5 threshold.